In [ ]:
# ================================
# SESSION
# ================================
from snowflake.snowpark.context import get_active_session
session = get_active_session()

# ================================
# CORE LIBRARIES
# ================================
import pandas as pd

# ================================
# SKLEARN + XGBOOST
# ================================
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

# ================================
# DATA SPLIT + METRICS
# ================================
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, roc_auc_score

# ================================
# SNOWFLAKE MODEL REGISTRY
# ================================
from snowflake.ml.registry import Registry

In [ ]:
# ================================
# LOAD DATA FROM SNOWFLAKE
# ================================
session.sql("USE DATABASE AI_CRISIS_LOANS").collect()
session.sql("USE SCHEMA LOAN_ANALYTICS").collect()

FEATURE_COLS = [
    "AGE",
    "ANNUAL_INCOME_USD",
    "INCOME_VOLATILITY_SCORE",
    "NET_WORTH_USD",
    "YEARS_IN_CURRENT_ROLE",
    "MONTHS_UNEMPLOYED_LAST3YR",
    "HAS_AI_CERT_FLAG",
    "HAS_COLLATERAL_FLAG",
    "DIGITAL_LITERACY_SCORE",
    "EDUCATION_SCORE",
    "SOCIAL_SUPPORT_SCORE_NUM"
]

TARGET_COL = "TARGET"

query = f"""
SELECT {', '.join(FEATURE_COLS)}, {TARGET_COL}
FROM LOAN_FEATURES_TABLE_CLEAN
LIMIT 200000
"""

df = session.sql(query).to_pandas()

# Fill nulls safely
df = df.fillna(0)

In [ ]:
# ================================
# TRAIN / TEST SPLIT
# ================================
X = df[FEATURE_COLS]
y = df[TARGET_COL]

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Train:", X_train.shape, "Test:", X_test.shape)

In [ ]:
# ================================
# MODEL PIPELINES
# ================================

rf_pipeline = Pipeline([
    ("model", RandomForestClassifier(
        n_estimators=300,
        max_depth=10,
        min_samples_split=10,
        min_samples_leaf=5,
        class_weight="balanced",
        random_state=42,
        n_jobs=-1
    ))
])

# Handle imbalance properly
scale_pos_weight = len(y_train[y_train==0]) / len(y_train[y_train==1])

xgb_pipeline = Pipeline([
    ("model", XGBClassifier(
        n_estimators=300,
        max_depth=8,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        scale_pos_weight=scale_pos_weight,
        random_state=42,
        eval_metric="logloss"
    ))
])

In [ ]:
# ================================
# TRAIN MODELS
# ================================
rf_pipeline.fit(X_train, y_train)
print("✅ Random Forest trained")

xgb_pipeline.fit(X_train, y_train)
print("✅ XGBoost trained")

In [ ]:
# ================================
# EVALUATION
# ================================
def evaluate(model, name):
    y_pred = model.predict(X_test)
    y_proba = model.predict_proba(X_test)[:, 1]

    acc = accuracy_score(y_test, y_pred)
    auc = roc_auc_score(y_test, y_proba)

    print(f"{name} → Accuracy: {acc:.4f} | AUC: {auc:.4f}")
    return acc, auc

rf_acc, rf_auc = evaluate(rf_pipeline, "Random Forest")
xgb_acc, xgb_auc = evaluate(xgb_pipeline, "XGBoost")

In [ ]:
# ================================
# SELECT BEST MODEL
# ================================
if xgb_auc > rf_auc:
    best_model = xgb_pipeline
    best_name = "XGBoost"
    best_auc = xgb_auc
    best_acc = xgb_acc
else:
    best_model = rf_pipeline
    best_name = "Random Forest"
    best_auc = rf_auc
    best_acc = rf_acc

print(f"\n🏆 Selected Model: {best_name}")
print(f"AUC: {best_auc:.4f}, Accuracy: {best_acc:.4f}")

In [ ]:
# ================================
# FEATURE IMPORTANCE
# ================================
model = best_model.named_steps["model"]

feat_imp = pd.DataFrame({
    "feature": FEATURE_COLS,
    "importance": model.feature_importances_
}).sort_values(by="importance", ascending=False)

print("\nTop Features:")
print(feat_imp.head(10))

In [ ]:
# ================================
# REGISTER MODEL
# ================================
registry = Registry(
    session=session,
    database_name="AI_CRISIS_LOANS",
    schema_name="LOAN_ANALYTICS"
)

sample_input = pd.DataFrame(X_train[:100], columns=FEATURE_COLS)

mv = registry.log_model(
    best_model,
    model_name="AI_CRISIS_DEFAULT_MODEL_PIPELINE",
    comment="Final clean pipeline (no leakage, no scaling)",
    metrics={"auc": float(best_auc), "accuracy": float(best_acc)},
    sample_input_data=sample_input,
    target_platforms=["WAREHOUSE"]
)

print("\n✅ Model registered!")
print("Model:", best_name)
print("Version:", mv.version_name)